# Hyperparameter Tuning — Random Forest URL Classifier
**Google Colab | Dataset 200.000 Balanced | RandomizedSearchCV + StratifiedKFold**

---
Notebook ini mencari kombinasi hyperparameter terbaik untuk model Random Forest
menggunakan **RandomizedSearchCV** (lebih efisien dari GridSearch untuk ruang besar).

Output notebook ini adalah file `best_params.json` yang langsung dipakai di notebook training.

**Alur:**
1. Load dataset → ambil 200.000 sampel balanced (100K aman + 100K porno)
2. Definisikan ruang pencarian hyperparameter
3. Jalankan RandomizedSearchCV (30 iterasi × 3-fold CV)
4. Simpan hyperparameter terbaik ke `best_params.json`

## Cell 1 — Install Library

In [ ]:
# Semua library yang dibutuhkan sudah tersedia di Colab,
# kecuali seaborn yang kadang perlu diupdate.
!pip install -q --upgrade scikit-learn seaborn
print('Library siap.')

## Cell 2 — Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, StratifiedKFold
)
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score, make_scorer
)

print('Library berhasil diimport.')
print(f'scikit-learn version: {__import__("sklearn").__version__}')

## Cell 3 — Mount Google Drive & Konfigurasi Path
Sesuaikan `DATASET_PATH` dengan lokasi file CSV dataset kamu di Google Drive.

**Format dataset yang diharapkan** — CSV dengan kolom:
```
url, label, domain_length, digit_count, dot_count, delimiter_count,
suspicious_word_count, digit_letter_ratio, max_sequential_digits
```
> `label`: 0 = URL aman, 1 = URL pornografi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ================================================================
# SESUAIKAN PATH INI
# ================================================================
DATASET_PATH = '/content/drive/MyDrive/Tugas Akhir/Dataset/dataset_url.csv'
SAVE_PATH    = '/content/drive/MyDrive/Tugas Akhir/Model/RF/'
# ================================================================

os.makedirs(SAVE_PATH, exist_ok=True)

FEATURE_COLS = [
    'domain_length',
    'digit_count',
    'dot_count',
    'delimiter_count',
    'suspicious_word_count',
    'digit_letter_ratio',
    'max_sequential_digits'
]
LABEL_COL = 'label'

df = pd.read_csv(DATASET_PATH)
print(f'Dataset berhasil dimuat.')
print(f'Shape : {df.shape[0]:,} baris x {df.shape[1]} kolom')
print(f'Kolom : {list(df.columns)}')
print(f'\nDistribusi label:')
print(df[LABEL_COL].value_counts())
df.head()

## Cell 4 — Sampling 200.000 Data Balanced

Kenapa 200.000?
- Random Forest dengan 7 fitur mencapai plateau performa di sekitar angka ini
- Cukup untuk CNN-1D menunjukkan kemampuannya
- Dataset yang sama dipakai untuk **kedua model** agar perbandingan adil

Jika dataset asli kurang dari 100K per kelas, sesuaikan `N_PER_CLASS`.

In [ ]:
N_PER_CLASS = 100_000  # 100K aman + 100K porno = 200K total

df_safe = df[df[LABEL_COL] == 0]
df_porn = df[df[LABEL_COL] == 1]

print(f'Total URL aman      : {len(df_safe):,}')
print(f'Total URL pornografi: {len(df_porn):,}')

# Pastikan jumlah data cukup
assert len(df_safe) >= N_PER_CLASS, f'Data aman kurang dari {N_PER_CLASS:,}'
assert len(df_porn) >= N_PER_CLASS, f'Data porno kurang dari {N_PER_CLASS:,}'

df_safe_s = df_safe.sample(n=N_PER_CLASS, random_state=42)
df_porn_s = df_porn.sample(n=N_PER_CLASS, random_state=42)

# Gabung dan acak
df_bal = pd.concat([df_safe_s, df_porn_s]).sample(frac=1, random_state=42).reset_index(drop=True)

# Bersihkan duplikat dan missing value
n_before = len(df_bal)
df_bal = df_bal.drop_duplicates(subset=FEATURE_COLS + [LABEL_COL])
df_bal = df_bal.dropna(subset=FEATURE_COLS + [LABEL_COL]).reset_index(drop=True)
print(f'\nSetelah sampling   : {n_before:,} baris')
print(f'Setelah bersih     : {len(df_bal):,} baris')
print(f'Distribusi akhir   :\n{df_bal[LABEL_COL].value_counts()}')
df_bal[FEATURE_COLS].describe()

## Cell 5 — Split Data Training & Testing

80% untuk training (dipakai CV), 20% untuk evaluasi akhir.
`stratify=y` memastikan proporsi kelas tetap sama di train dan test.

In [ ]:
X = df_bal[FEATURE_COLS].values.astype(np.float32)
y = df_bal[LABEL_COL].values

# Tidak ada scaling — RF tidak membutuhkan normalisasi fitur
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set : {len(X_train):,} baris')
print(f'Testing set  : {len(X_test):,} baris')
print(f'Train label  : {np.bincount(y_train)}')
print(f'Test label   : {np.bincount(y_test)}')

## Cell 6 — Definisi Ruang Pencarian Hyperparameter

Penjelasan setiap hyperparameter:

| Parameter | Pengaruh |
|-----------|----------|
| `n_estimators` | Jumlah pohon. Lebih banyak = lebih stabil, tapi lebih lambat |
| `max_depth` | Kedalaman pohon. `None` = tanpa batas (risiko overfitting) |
| `max_features` | Jumlah fitur yang dipertimbangkan di setiap split. `sqrt`/`log2` = default RF |
| `min_samples_split` | Minimum sampel untuk membagi node. Tinggi = regularisasi lebih kuat |
| `min_samples_leaf` | Minimum sampel di daun pohon. Tinggi = model lebih smooth |
| `class_weight` | `balanced` = bobot proporsional kelas, cocok untuk imbalanced data |

In [ ]:
param_dist = {
    'n_estimators'     : [100, 150, 200, 300, 500],
    'max_depth'        : [5, 8, 10, 15, 20, None],
    'max_features'     : ['sqrt', 'log2', 0.3, 0.5],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf' : [1, 2, 4, 8],
    'class_weight'     : ['balanced', 'balanced_subsample'],
}

# Hitung total kombinasi
total_kombinasi = 1
for v in param_dist.values():
    total_kombinasi *= len(v)

print('Ruang pencarian hyperparameter:')
for k, v in param_dist.items():
    print(f'  {k:<22}: {v}')
print(f'\nTotal kombinasi : {total_kombinasi:,}')
print(f'Yang dicoba     : 30 iterasi (RandomizedSearchCV)')
print(f'Cross-validation: 3-fold StratifiedKFold')

## Cell 7 — Jalankan RandomizedSearchCV

**Metrik optimasi: F1-Score** (bukan accuracy) karena:
- F1 menyeimbangkan Precision dan Recall
- Lebih relevan untuk deteksi konten — kita ingin menangkap porno (Recall tinggi)
  tanpa memblokir terlalu banyak URL aman (Precision juga tinggi)

> Estimasi waktu: **15–40 menit** tergantung hardware Colab yang didapat.

In [ ]:
# StratifiedKFold: menjaga proporsi kelas di setiap fold
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Optimalkan F1 untuk kelas 1 (pornografi)
scorer = make_scorer(f1_score, pos_label=1)

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

search = RandomizedSearchCV(
    estimator           = rf_base,
    param_distributions = param_dist,
    n_iter              = 30,        # Coba 30 kombinasi acak
    cv                  = cv,
    scoring             = scorer,
    n_jobs              = -1,        # Gunakan semua CPU
    verbose             = 1,
    random_state        = 42,
    return_train_score  = True
)

print('Memulai RandomizedSearchCV...')
print(f'(30 iterasi × 3-fold = 90 kali training, setiap kali dengan ~{len(X_train)*2//3:,} data)')
print('-' * 50)

t0 = time.time()
search.fit(X_train, y_train)
durasi = time.time() - t0

print(f'\nSelesai dalam {durasi:.0f} detik ({durasi/60:.1f} menit)')
print(f'\nF1 terbaik (CV mean) : {search.best_score_:.4f}')
print(f'Hyperparameter terbaik:')
for k, v in search.best_params_.items():
    print(f'  {k:<22}: {v}')

## Cell 8 — Tampilkan Hasil Semua Iterasi

In [ ]:
results_df = pd.DataFrame(search.cv_results_)
results_df = results_df.sort_values('mean_test_score', ascending=False).reset_index(drop=True)

# Kolom yang relevan
cols_show = ['mean_test_score', 'std_test_score', 'mean_train_score']
param_cols = [c for c in results_df.columns if c.startswith('param_')]

display(results_df[cols_show + param_cols].head(10).round(4))

# Plot F1 semua iterasi
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(results_df)), results_df['mean_test_score'].values, color='#1E88E5', alpha=0.7)
ax.axhline(search.best_score_, color='red', linestyle='--', label=f'Terbaik: {search.best_score_:.4f}')
ax.set_xlabel('Iterasi')
ax.set_ylabel('F1-Score (CV)')
ax.set_title('F1-Score Setiap Iterasi RandomizedSearchCV')
ax.legend()
plt.tight_layout()
plt.show()

## Cell 9 — Evaluasi Model Terbaik pada Test Set

In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

print('Evaluasi pada TEST SET (20% data yang tidak digunakan saat tuning):')
print('=' * 50)
print(f'  Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
print(f'  Precision : {precision_score(y_test, y_pred):.4f}')
print(f'  Recall    : {recall_score(y_test, y_pred):.4f}')
print(f'  F1-Score  : {f1_score(y_test, y_pred):.4f}')
print('=' * 50)

# Cek overfitting: bandingkan train vs test F1
y_pred_train = best_model.predict(X_train)
f1_train = f1_score(y_train, y_pred_train)
f1_test  = f1_score(y_test, y_pred)
gap = f1_train - f1_test
print(f'\nCek Overfitting:')
print(f'  F1 Train  : {f1_train:.4f}')
print(f'  F1 Test   : {f1_test:.4f}')
print(f'  Gap       : {gap:.4f}', end=' ')
if gap < 0.02:
    print('✅ Baik (tidak overfitting)')
elif gap < 0.05:
    print('⚠️ Sedikit overfitting')
else:
    print('❌ Overfitting — coba kurangi max_depth atau tambah min_samples_leaf')

## Cell 10 — Simpan Hyperparameter Terbaik ke JSON
File ini akan dibaca otomatis oleh notebook training.

In [ ]:
best_params = search.best_params_.copy()

# Simpan ke Drive
json_path = SAVE_PATH + 'best_params.json'
with open(json_path, 'w') as f:
    json.dump(best_params, f, indent=2)

print(f'✅ Tersimpan: {json_path}')
print('\nIsi best_params.json:')
print(json.dumps(best_params, indent=2))

# Simpan juga seluruh hasil tuning
results_df.to_csv(SAVE_PATH + 'tuning_results.csv', index=False)
print(f'\nHasil semua iterasi tersimpan: {SAVE_PATH}tuning_results.csv')

print('\n--- SELESAI ---')
print('Langkah berikutnya: Buka notebook rf_training_v2.ipynb')